In [4]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import math

In [5]:
#1.Positional Encoding

class PositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=100):
        super().__init__()

        pe = torch.zeros(max_len,d_model)

        for pos in range(max_len):
            for i in range(0,d_model,2):

                pe[pos,i] = math.sin(pos/(10000 ** ((2*i)/d_model)))

                if i + 1 < d_model :
                    pe[pos, i+1] = math.cos(pos / (10000 ** ((2*i)/d_model)))
        
        self.pe = pe.unsqueeze(0)


    def forward(self,x):
        x = x + self.pe[:,:x.size(1)]
        return x

In [6]:
#scaled dot product attention 
def attention (Q,K,V):
    d_k = Q.size(-1)
    scores = torch.matmul(Q,K.transpose(-2,-2)/math.sqrt(d_k))
    weights = F.softmax(scores,dim = -1)
    output = torch.matmul(weights,V)
    return output

In [7]:
#Multi Head attention

class MultiHeadAttention(nn.Module):

    def __init__(self,d_model,heads):
        super().__init__()
 
        self.heads = heads
        self.d_k  = d_model // heads

        self.q_linear = nn.Linear(d_model,d_model)
        self.k_linear = nn.Linear(d_model,d_model)
        self.v_linear = nn.Linear(d_model,d_model)

        self.fc_out = nn.Linear(d_model,d_model)

    def forward(self,x):

        batch_size, seq_len, d_model = x.shape

        Q = self.q_linear(x)
        K = self.k_linear(x)
        v = self.v_linear(x)

        Q = Q.view(batch_size,seq_len,self.heads,self.d_k)
        K = K.view(batch_size,seq_len,self.heads,self.d_k)
        V = V.view(batch_size,seq_len,self.heads,self.d_k)

        Q = Q.transpose(1,2)
        K = K.transpose(1,2)
        V = V.transpose(1,2)

        out = attention(Q,K,V)

        out = out.transpose(1,2).contiguous().view(batch_size,seq_len,d_model)

        out = self.fc_out(out)

        return out

